In [1]:
!pip install pyspark


In [2]:
!apt-get install openjdk-11-jdk

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openjdk-11-jdk is already the newest version (11.0.28+6-1ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Import access_log.txt

In [4]:
from pyspark import SparkContext
import re

In [5]:
# SparkContext: The main object that connects your code to Spark’s engine. It lets you load data, create RDDs, and run distributed operations.

# "local": Tells Spark to run in local mode, using your computer's resources (not a cluster). Great for development and small-scale testing.

# "LogAnalysis": This is the application name. It helps identify your job in Spark’s UI or logs. You can name it anything relevant.

sc = SparkContext("local", "LogAnalysis")

In [6]:
# Load log file
logs = sc.textFile("/content/access_log.txt")

# First 10 lines of our access_log text file
print("logs:\n")
for line in logs.take(10):
    print(line)

logs:

2025-10-18 12:00:00 INFO User107 accessed /home
2025-10-18 12:00:01 INFO User105 accessed /home
2025-10-18 12:00:02 ERROR User101 failed to load /dashboard
2025-10-18 12:00:03 INFO User112 accessed /login
2025-10-18 12:00:04 INFO User118 accessed /home
2025-10-18 12:00:05 INFO User120 accessed /login
2025-10-18 12:00:06 INFO User113 accessed /home
2025-10-18 12:00:07 INFO User114 accessed /home
2025-10-18 12:00:08 INFO User110 accessed /login
2025-10-18 12:00:09 ERROR User118 failed to load /login


In [7]:
# Function to parse log lines
def parse_line(line):
    match = re.match(r"(\S+ \S+) (\S+) (\S+) .* (/[\w]+)", line)
    if match:
        timestamp, level, user, endpoint = match.groups()
        return (user, level, endpoint)
    else:
        return ("UNKNOWN", "UNKNOWN", "UNKNOWN")

# Parse logs
parsed_logs = logs.map(parse_line)

# First 10 lines of our access_log text file
print("access_log:\n")
for line in parsed_logs.take(10):
    print(line)

access_log:

('User107', 'INFO', '/home')
('User105', 'INFO', '/home')
('User101', 'ERROR', '/dashboard')
('User112', 'INFO', '/login')
('User118', 'INFO', '/home')
('User120', 'INFO', '/login')
('User113', 'INFO', '/home')
('User114', 'INFO', '/home')
('User110', 'INFO', '/login')
('User118', 'ERROR', '/login')


In [8]:
# Task 1: Count log entries per user
user_activity = parsed_logs.map(lambda x: (x[0], 1)).reduceByKey(lambda a, b: a + b)

print("Count log entries per user:\n")
for line in user_activity.take(5):
    print(line)

Count log entries per user:

('User107', 7)
('User105', 4)
('User101', 6)
('User112', 3)
('User118', 6)


In [9]:
# Task 2: Count ERRORs per endpoint
error_counts = parsed_logs.filter(lambda x: x[1] == "ERROR") \
                          .map(lambda x: (x[2], 1)) \
                          .reduceByKey(lambda a, b: a + b)

print("Error Counts per Endpoint:\n")
for endpoint, count in error_counts.collect():
    print(f"{endpoint}: {count}")

Error Counts per Endpoint:

/dashboard: 12
/login: 13
/settings: 5
/home: 7
/profile: 3


In [10]:
# Task 3: Top 5 most active users
top_users = user_activity.takeOrdered(5, key=lambda x: -x[1])

# Print results
print("Top 5 Active Users:\n")
for user, count in top_users:
    print(f"{user}: {count}")

Top 5 Active Users:

User120: 9
User108: 8
User107: 7
User101: 6
User118: 6


In [11]:
# Task 4: Count INFOs per endpoint

# Your task: Write PySpark code that counts how many times each endpoint was accessed with an INFO log level.
# Hint: This is very similar to how error_counts was implemented.

info_counts = parsed_logs.filter(lambda x: x[1] == "INFO") \
                          .map(lambda x: (x[2], 1)) \
                          .reduceByKey(lambda a, b: a + b)

print("INFO Counts per Endpoint:\n")
for endpoint, count in info_counts.collect():
    print(f"{endpoint}: {count}")

INFO Counts per Endpoint:

/home: 15
/login: 11
/settings: 11
/profile: 14
/dashboard: 9
